In [0]:
class Bronze_constructors():
    main_path="/Volumes/formula1_race/default/formula1/"
    bronze_path = "formula1_race_project/bronze" 
    
    def __init__(self,folder_name,source):
        self.folder_name=folder_name
        self.source=source
    
    def get_schema(self):
        schema = '''constructorId INT NOT NULL,
            constructorRef STRING NOT NULL,
            name STRING NOT NULL,
            nationality STRING,
            url STRING NOT NULL
            '''

        return schema

    def read_data(self):
        df= (spark.readStream
             .format("json")
             .schema(self.get_schema())
             .option("maxFilesPerTrigger", 1)
             #.option("rowsPerSecond", 1000)
             .option('multiline','true')
             .load(f"{self.main_path}/formula1_source/{self.folder_name}")
             )
        return df
        
    def process(self):
        print(f"\nStarting Bronze constructor Stream...", end='')
        readDF = self.read_data()
        from pyspark.sql.functions import current_timestamp,lit
        readDF= (readDF.withColumn('ConstructorsIngestionDate',current_timestamp())
                 .withColumn('source',lit(self.source))
                 )
        sQuery =  ( readDF.writeStream
                            .queryName("bronze-ingestion-constructor")
                            .option("checkpointLocation", f"{self.main_path}/{self.bronze_path}/{self.folder_name}/checkpoint")
                            .outputMode("append")#full load so we need to use overwrite or complete mode n=but it is not supported by community edition
                            #.option('path',f"{self.main_path}/{self.bronze_path}/{self.folder_name}")
                            .trigger(availableNow=True)
                            .toTable('formula1_race.bronze.constructors')
                               
                    ) 
        print("Done")
        return sQuery   


In [0]:
from pyspark.sql.streaming import StreamingQueryListener

class MyListener(StreamingQueryListener):
    def onQueryStarted(self, event):
        print(f"Query started: {event.id}")

    def onQueryProgress(self, event):
        print(f"Batch ID: {event.progress['batchId']}")
        print(f"Rows Read: {event.progress['numInputRows']}")
        print(f"Duration (ms): {event.progress['durationMs']}")
        print(f"Rows/sec: {event.progress['processedRowsPerSecond']}")

    def onQueryTerminated(self, event):
        print(f"Query terminated: {event.id}")

spark.streams.addListener(MyListener())

In [0]:
source='Ergast API'
Bronze_constructors_instance = Bronze_constructors("constructors",source)
Squery_Bronze_constructors = Bronze_constructors_instance.process()
Squery_Bronze_constructors.awaitTermination()
print("Successfully bronze-ingestion-constructors stream in running")
Squery_Bronze_constructors.stop()